# Tools
- Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
 1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
 2. A function or co-routine to execute

In [5]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("Why do parrots talk?")
response.content

'Parrots “talk” because they’re exceptional mimics, social animals, and intelligent communicators all rolled into one. Below is a quick‑look at the science, the behavior, and the practical side of why a parrot will chatter away in your living room.\n\n---\n\n## 1. The Biological Basis\n\n| Feature | How it Helps Parrots Talk |\n|---------|---------------------------|\n| **Syrinx (vocal organ)** | The parrot’s syrinx is located at the base of the trachea, giving them incredible control over pitch, volume, and timbre. |\n| **Vocal Learning** | Like humans, some parrots can *learn* sounds by listening and then reproducing them. This ability is rare among birds (others include songbirds, hummingbirds, and some owls). |\n| **Brain‑to‑Syrinx Circuitry** | The avian brain has a specialized “vocal learning” pathway that links auditory perception directly to the syrinx, allowing parrots to fine‑tune their mimicry. |\n| **Large Auditory Cortex** | Parrots have a highly developed region for proce

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a loction"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])


In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks: "What\'s the weather like in Boston?" We have a function get_weather that can get weather at a location. We should call it.', 'tool_calls': [{'id': 'fc_98514644-8e9e-4c5d-86a9-012b64b758c9', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 128, 'total_tokens': 183, 'completion_time': 0.057647137, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.006377783, 'prompt_tokens_details': None, 'queue_time': 0.340999495, 'total_time': 0.06402492}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_34cb00738d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0b75a-a594-7303-9a0c-b3a13eb59c26-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_98514644-8e9e-4c5d-86a9-012b64b75

### Tool execution loop

In [4]:
# model generates tool calls
messages = [{"role":"user","content":"What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Execute tools and collect result
for tool_call in ai_msg.tool_calls:
    tool_res = get_weather.invoke(tool_call)
    messages.append(tool_res)

# pass result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response)    

content='It’s a bright, sunny day in Boston today—perfect weather for a stroll along the Charles River or a quick bite at a local café! If you’re planning to head out, you might want to bring a light jacket for the cooler evenings. Enjoy the sunshine!' additional_kwargs={'reasoning_content': 'We got a response: "It\'s sunny in Boston". The user asked: "What\'s the weather in Boston?" The assistant responded with weather info. That seems fine. But maybe we want to provide more details: temperature, humidity, etc. But the function returned only "It\'s sunny in Boston". We can elaborate. But the instruction says we can use the function to get the weather. We did. We can now provide a friendly answer. Let\'s do that.'} response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 156, 'total_tokens': 312, 'completion_time': 0.163549465, 'completion_tokens_details': {'reasoning_tokens': 93}, 'prompt_time': 0.00869161, 'prompt_tokens_details': None, 'queue_time': 0.344550488,